# 01 · Preprocessing — Gaza, 2024-05-03

Produces the training stacks: Sentinel-1 period statistics from Earth Engine,
with UNOSAT damage labels burned on from the PWTT benchmark CSVs.

**Output:** one GeoTIFF per tile, 29 bands.

| bands | content |
|---|---|
| 1–12 | reference period, 12 months pre-onset: per orbit `VV_mean, VV_sd, VH_mean, VH_sd, n, valid` |
| 13–24 | inference period, 1 month from the UNOSAT date, same layout |
| 25–29 | `fid`, `label`, `area`, `gx`, `gy` |

Those 12 channels per period are exactly what Eq. 1–3 of the paper need — mean,
standard deviation, scene count, per orbit and polarisation.

In [ ]:
!pip install -q earthengine-api geopandas rasterio

In [ ]:
import json, glob, math, os
import ee, numpy as np, pandas as pd, geopandas as gpd, rasterio
import matplotlib.pyplot as plt
from rasterio.features import rasterize
from rasterio.warp import transform as warp_transform
from shapely.geometry import shape
from sklearn.metrics import precision_recall_curve, roc_auc_score

## Constants

The patch geometry follows the paper. Its aggregation unit is the 500 m UNOSAT
cell, which at Sentinel-1's native 10 m/px is **50×50 px** — that is the core of a
sample and the only region the loss will ever see. Around it sits a 15 px halo,
because the PWTT smooths with kernels of radius up to 150 m; without it the
largest kernel would read padding exactly where supervision happens. So the
network input is 80×80, and 80 = 5·2⁴ divides cleanly through four poolings.

In [ ]:
PIX_M    = 10                      # Sentinel-1 GRD IW resolution
CORE_PX  = 50                      # 500 m = one UNOSAT rapid-assessment cell
HALO_PX  = 15                      # 150 m = largest PWTT smoothing kernel
PATCH_PX = CORE_PX + 2 * HALO_PX   # 80
GRID_M   = 500
GRID_CRS = 'EPSG:3857'             # the grid CSV keys cells as "gx,gy" in this projection

ORBITS = 2                         # one ascending + one descending
POLS   = ('VV', 'VH')
IN_CH  = ORBITS * 6                # per orbit: VV_mean, VV_sd, VH_mean, VH_sd, n, valid

BAND_ORDER = (
    [f'{p}_o{o}_{b}' for p in ('pre','post') for o in range(ORBITS)
     for b in ('VV_mean','VV_sd','VH_mean','VH_sd','n','valid')]
    + ['fid', 'label', 'area', 'gx', 'gy'])

ONSET, ASSESSMENT = '2023-10-10', '2024-05-03'   # 12-month reference / 1-month inference
ENL = 4.4                                        # equivalent number of looks, IW GRD

print(PATCH_PX, 'px in ->', CORE_PX, 'px core =', CORE_PX*PIX_M, 'm |',
      IN_CH, 'channels per branch |', len(BAND_ORDER), 'bands out')

## 1 · The dependent variable

`class` is the label: 1 where the footprint intersects a UNOSAT damage
annotation. `max_change` is the PWTT's raw statistic max|t| pooled over the
footprint, and `k50/k100/k150` the same from the convolved layers — the paper's
damage raster is their mean (Section 2.2).

In [ ]:
FOOTPRINTS_CSV = 'Gaza_20240503_1_footprints.csv'   # upload to /content or point at Drive
GRID_CSV       = 'Gaza_20240503_1_grid.csv'

LAYERS = ['max_change', 'k50', 'k100', 'k150']

fp = pd.read_csv(FOOTPRINTS_CSV,
                 usecols=['area','class','damage_pts','.geo'] + LAYERS)
fp['fid']  = np.arange(1, len(fp) + 1)
fp['pwtt'] = fp[LAYERS].mean(axis=1)
fp = gpd.GeoDataFrame(fp.drop(columns='.geo'),
                      geometry=[shape(json.loads(g)) for g in fp['.geo']],
                      crs='EPSG:4326')

grid = pd.read_csv(GRID_CSV)
grid[['gx','gy']] = [list(map(int, i.split(','))) for i in grid['system:index']]
grid['pwtt'] = grid[LAYERS].mean(axis=1)

print(f'{len(fp):,} footprints, {100*fp["class"].mean():.1f}% damaged '
      f'({100*np.average(fp["class"], weights=fp["area"]):.1f}% by area)')
print(f'{len(grid):,} grid cells, {100*grid["class"].mean():.1f}% damaged')

### Sanity check first

Pooled per footprint, area-weighted, thresholded at T > 2.9, the mean of the four
layers should reproduce Table 3's row for 2024-05-03: **F1 77.75 · P 73.82 ·
R 82.13 · AUC 82.69**. If this doesn't land near those numbers, nothing
downstream is worth running.

In [ ]:
def metrics(score, y, w, threshold=None):
    """Area-weighted building-level accuracy; threshold from a PR curve if not given."""
    score, y, w = np.asarray(score), np.asarray(y, float), np.asarray(w, float)
    if threshold is None:
        p, r, t = precision_recall_curve(y, score, sample_weight=w)
        f1 = 2*p*r / np.maximum(p+r, 1e-9)
        threshold = t[int(np.nanargmax(f1[:-1]))]
    pred = (score > threshold).astype(float)
    tp = (w*pred*y).sum();      fp_ = (w*pred*(1-y)).sum()
    fn = (w*(1-pred)*y).sum();  tn  = (w*(1-pred)*(1-y)).sum()
    P, R = tp/max(tp+fp_,1e-9), tp/max(tp+fn,1e-9)
    return dict(threshold=round(float(threshold),3),
                AUC=100*roc_auc_score(y, score, sample_weight=w),
                Accuracy=100*(tp+tn)/(tp+tn+fp_+fn),
                F1=100*2*P*R/max(P+R,1e-9), Precision=100*P, Recall=100*R)

for name, s in [('raw max|t|', fp['max_change']), ('mean of 4 layers', fp['pwtt'])]:
    m = metrics(s, fp['class'], fp['area'])
    print(f'{name:20s} ' + '  '.join(f'{k}={m[k]:6.2f}'
          for k in ['AUC','Accuracy','F1','Precision','Recall']))

m = metrics(fp['pwtt'], fp['class'], fp['area'], threshold=2.9)
print('\nT > 2.9:', {k: round(v,2) for k,v in m.items()})
print('Table 3: AUC 82.69  F1 77.75  P 73.82  R 82.13')

Smoothing is worth ~9 AUC points on its own — which is why it stays a fixed
layer on the network output rather than something the decoder must rediscover.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for col, a, title in [('max_change', ax[0], 'raw max|t|'), ('pwtt', ax[1], 'mean of 4 layers')]:
    for c, lab in [(0,'undamaged'), (1,'damaged')]:
        a.hist(fp.loc[fp['class']==c, col], bins=80, range=(0,12), alpha=.55,
               density=True, label=lab)
    a.axvline(2.9, color='k', ls='--', lw=1); a.set_title(title); a.set_xlabel('T'); a.legend()
plt.tight_layout(); plt.show()

## 2 · Sentinel-1 export

Section 2.1.1: GRD IW, Lee speckle filter in the power domain, log transform, and
scenes kept **separated by relative orbit number** so the look angle is constant
within each comparison.

One ascending and one descending orbit are picked — the ones with most reference
scenes. The PWTT takes the max over however many orbits exist; a CNN needs a fixed
channel count, so cities with more orbits lose that extra information.

In [ ]:
ee.Authenticate()
ee.Initialize(project='satelite-data-504215')

In [ ]:
def lee_filter(img, size=3):
    """Shrink the local mean toward the observation by the local coefficient of
    variation — edge-preserving speckle suppression (Lee et al. 1994)."""
    k = ee.Kernel.square(size // 2, 'pixels')
    mean = img.reduceNeighborhood(ee.Reducer.mean(), k)
    var  = img.reduceNeighborhood(ee.Reducer.variance(), k)
    ci2 = var.divide(mean.pow(2).max(1e-8))
    w = ee.Image.constant(1.0).subtract(ee.Image.constant(1.0/ENL).divide(ci2.max(1e-8))).max(0)
    return mean.add(w.multiply(img.subtract(mean))).rename(img.bandNames())

def s1_collection(aoi):
    col = (ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')
           .filterBounds(aoi)
           .filter(ee.Filter.eq('instrumentMode', 'IW'))
           .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
           .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
           .select(list(POLS)))
    return col.map(lambda im: lee_filter(im).max(1e-8).log10().multiply(10)
                                .copyProperties(im, im.propertyNames()))

In [ ]:
minx, miny, maxx, maxy = fp.total_bounds
aoi = ee.Geometry.Rectangle([minx-.01, miny-.01, maxx+.01, maxy+.01])

col = s1_collection(aoi)
ref_start, ref_end = ee.Date(ONSET).advance(-12,'month'), ee.Date(ONSET)

orbits = []
for direction in ['ASCENDING', 'DESCENDING']:
    sub = (col.filterDate(ref_start, ref_end)
              .filter(ee.Filter.eq('orbitProperties_pass', direction)))
    counts = {o: sub.filter(ee.Filter.eq('relativeOrbitNumber_start', o)).size().getInfo()
              for o in sub.aggregate_array('relativeOrbitNumber_start').distinct().getInfo()}
    best = max(counts, key=counts.get)
    orbits.append(best)
    print(f'{direction:11s} scenes per relative orbit {counts}  -> using {best}')

In [ ]:
def period_stats(orbit, start, end, prefix, i):
    """Eq. 1-2 for one (orbit, period): per-pixel mean and sd of backscatter, plus n."""
    sub = (col.filter(ee.Filter.eq('relativeOrbitNumber_start', orbit))
              .filterDate(start, end))
    n = sub.size()
    mean = sub.mean().rename([f'{prefix}_o{i}_{p}_mean' for p in POLS])
    sd   = sub.reduce(ee.Reducer.stdDev()).rename([f'{prefix}_o{i}_{p}_sd' for p in POLS])
    bands = []
    for p in POLS:
        bands += [mean.select(f'{prefix}_o{i}_{p}_mean'), sd.select(f'{prefix}_o{i}_{p}_sd')]
    bands += [ee.Image.constant(n).float().rename(f'{prefix}_o{i}_n'),
              ee.Image.constant(ee.Number(n).gt(2)).float().rename(f'{prefix}_o{i}_valid')]
    return ee.Image.cat(bands).unmask(0)

inf_start, inf_end = ee.Date(ASSESSMENT), ee.Date(ASSESSMENT).advance(1,'month')
stack = ee.Image.cat(
    [period_stats(o, ref_start, ref_end, 'pre',  i) for i, o in enumerate(orbits)] +
    [period_stats(o, inf_start, inf_end, 'post', i) for i, o in enumerate(orbits)]
).toFloat().clip(aoi)

print(stack.bandNames().getInfo())

In [ ]:
# UTM keeps pixels square and exactly 10 m
zone = int((minx + maxx)/2 + 180)//6 + 1
CRS = f'EPSG:{32600 + zone if (miny+maxy)/2 >= 0 else 32700 + zone}'

# tile the AOI; patches get cut locally, so export few large tiles
TILE_PX = 2048
side = TILE_PX * PIX_M
dlat = side / 111_320
dlon = side / (111_320 * math.cos(math.radians((miny + maxy)/2)))

tasks, i = [], 0
lon = minx
while lon < maxx:
    lat = miny
    while lat < maxy:
        region = ee.Geometry.Rectangle([lon, lat, min(lon+dlon, maxx), min(lat+dlat, maxy)])
        t = ee.batch.Export.image.toDrive(
            image=stack, description=f'gaza_{i:03d}_s1', folder='s1_bda',
            fileNamePrefix=f'gaza_{i:03d}_s1', region=region, scale=PIX_M,
            crs=CRS, maxPixels=1e10, fileFormat='GeoTIFF')
        t.start(); tasks.append(t); i += 1
        lat += dlat
    lon += dlon
print(f'launched {len(tasks)} export tasks in {CRS}')

In [ ]:
from collections import Counter
print(Counter(t.status()['state'] for t in tasks))   # re-run until all COMPLETED

## 3 · Burning the labels on

Earth Engine exported imagery only; labels are rasterised locally, which avoids
uploading a 228k-feature asset and keeps the CSV geometry authoritative.

Footprints are burned `all_touched=True`, **largest first**, so small buildings
overwrite large ones where they share a pixel. 31 % of the paper's footprints are
smaller than one Sentinel-1 pixel; the other ordering lets a warehouse swallow
its neighbours.

In [ ]:
TILE_DIR = '/content/drive/MyDrive/s1_bda'   # where the exports landed

def add_labels(ref_tif, out_tif):
    with rasterio.open(ref_tif) as src:
        profile, s1 = src.profile.copy(), src.read()
        assert s1.shape[0] == 2*IN_CH, f'{ref_tif}: {s1.shape[0]} bands, expected {2*IN_CH}'

        g = fp.to_crs(src.crs)
        g = g.assign(_a=g.geometry.area).sort_values('_a', ascending=False)
        burn = lambda field: rasterize(
            zip(g.geometry, g[field]), out_shape=(src.height, src.width),
            transform=src.transform, fill=0, all_touched=True, dtype='float64'
        ).astype('float32')

        # 500 m cell indices in EPSG:3857, so they join to the grid CSV directly
        rows, cols = np.meshgrid(np.arange(src.height), np.arange(src.width), indexing='ij')
        xs, ys = rasterio.transform.xy(src.transform, rows.ravel(), cols.ravel())
        x, y = warp_transform(src.crs, GRID_CRS, list(xs), list(ys))
        gx = np.floor(np.array(x)/GRID_M).reshape(rows.shape).astype('float32')
        gy = np.floor(np.array(y)/GRID_M).reshape(rows.shape).astype('float32')

        out = np.concatenate([s1, burn('fid')[None], burn('class')[None],
                              burn('area')[None], gx[None], gy[None]]).astype('float32')

    profile.update(count=out.shape[0], dtype='float32')
    with rasterio.open(out_tif, 'w', **profile) as dst:
        dst.write(out)
        dst.descriptions = tuple(BAND_ORDER)
    return out_tif

labelled = []
for ref in sorted(glob.glob(f'{TILE_DIR}/gaza_*_s1.tif')):
    out = add_labels(ref, ref.replace('_s1.tif', '.tif'))
    with rasterio.open(out) as s:
        fid = s.read(BAND_ORDER.index('fid') + 1).astype(int)
    ids, counts = np.unique(fid[fid > 0], return_counts=True)
    print(f'{os.path.basename(out)}  {len(ids):,} footprints, '
          f'median {np.median(counts):.0f} px, {100*(counts==1).mean():.0f}% single-pixel')
    labelled.append(out)

## 4 · Check one tile

Middle panel is the PWTT statistic recomputed from the exported bands (Eq. 3–4).
If it doesn't broadly agree with the labels on the right, the reference period or
the orbit selection is wrong and there is no point training on this.

In [ ]:
with rasterio.open(labelled[0]) as src:
    arr = src.read().astype('float32')

def pwtt_T(a):
    """Eq. 3-4: t per orbit and polarisation, T = max|t|."""
    ts = []
    for o in range(ORBITS):
        b = o*6
        n0, n1 = np.maximum(a[b+4], 2), np.maximum(a[IN_CH+b+4], 2)
        valid = a[b+5] * a[IN_CH+b+5]
        for p in range(2):
            m0, s0 = a[b+2*p], a[b+2*p+1]
            m1, s1_ = a[IN_CH+b+2*p], a[IN_CH+b+2*p+1]
            ts.append(np.abs((m0-m1) / np.sqrt(np.maximum(s0**2/n0 + s1_**2/n1, 1e-6))) * valid)
    return np.max(ts, axis=0)

T = pwtt_T(arr)
fid = arr[BAND_ORDER.index('fid')]; lab = arr[BAND_ORDER.index('label')]

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(arr[IN_CH], cmap='gray');                   ax[0].set_title('post VV mean (dB)')
im = ax[1].imshow(T, vmin=2.5, vmax=5, cmap='turbo');    ax[1].set_title('PWTT T, recomputed')
plt.colorbar(im, ax=ax[1], fraction=.04)
ax[2].imshow(np.where(fid > 0, lab, np.nan), cmap='bwr', vmin=0, vmax=1)
ax[2].set_title('UNOSAT label')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

### Does the recomputed T match the author's?

The CSV's `max_change` came from the author's own Earth Engine code. Pooling the
recomputed raster over the same footprints and correlating is the strongest
available check on this preprocessing. Expect high but not perfect — two orbits
here, all available orbits there.

In [ ]:
uniq, inv = np.unique(fid.astype(int).ravel(), return_inverse=True)
sums = np.bincount(inv, weights=T.ravel(), minlength=len(uniq))
cnts = np.bincount(inv, minlength=len(uniq))
mine = pd.DataFrame({'fid': uniq, 'T_new': sums/np.maximum(cnts,1)}).query('fid > 0')

cmp = fp[['fid','max_change','class','area']].merge(mine, on='fid')
print(f'{len(cmp):,} footprints in this tile | Pearson r = {cmp.T_new.corr(cmp.max_change):.3f}')
m = metrics(cmp.T_new, cmp['class'], cmp.area)
print('recomputed T, this tile:', {k: round(v,2) for k,v in m.items()})

## 5 · Patch count

Windows of 80×80 px stepping by 50, so the 500 m cores tile the city once while
the halos overlap. Cores with no building are dropped — under a footprint-pooled
loss they carry no supervision. This cell only counts them; the Dataset that
reads them lives in notebook 02.

In [ ]:
total = 0
for path in labelled:
    with rasterio.open(path) as src:
        fid = src.read(BAND_ORDER.index('fid') + 1)
        valid = src.read(BAND_ORDER.index('pre_o0_valid') + 1)
    h, w = fid.shape
    n = sum(1
            for r in range(0, h - CORE_PX + 1, CORE_PX)
            for c in range(0, w - CORE_PX + 1, CORE_PX)
            if fid[r:r+CORE_PX, c:c+CORE_PX].any()
            and valid[r:r+CORE_PX, c:c+CORE_PX].mean() >= 0.5)
    print(f'{os.path.basename(path)}: {n} usable patches')
    total += n
print(f'\n{total} patches -> notebook 02')

Done. Notebook 02 reads `TILE_DIR`.

One city and one date trains a Gaza detector, not a damage detector. The paper's
central finding is that models lose up to 29 % moving to a new city, so add
cities from Table 1 — re-run this notebook per city with its own onset date and
CSV from `github.com/oballinger/PWTT` — and hold entire cities out, never
patches.